## Tools


Models can request to call tools that performs tasks such as fetching data from database , searching the web or running the code . Tools are pairings of:

    1. A schema , including the name of the tool, a description, and/or argument definitions         {often a JSON schema}
    
    2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

response=model.invoke("hiii how are you ?")
response


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': "Hiii! I'm doing great, thank you for asking! 😊 \n\nHow are you doing today? Is there anything I can help you with?", 'extras': {'signature': 'EtUJCtIJARFNMg/stys65rgWs936jas2fNuAf5lhIvagy2/557S8NLu5USKrY6CQtwYJvCKB3acRVumyrl5mlys5Mc3MvHnvHgifScRb8/nhuNjRujiVqE/Zj1jZNSz/UUChbZNY2eZEHfgaH3Bhf8yJPsSzJqdpSBUfAmHauRkFy5k4GgeoW8SliKYUC5skBO2LMReKbcFYEyaeGONp/xwyAI3umgF6KjYUts+YNDeEMyUjPZ/QdWOViWvdrRkJhJt/aOOJnpKoY4E0+kYdz+drsSKdtV2wLuf2bUJABfgPUWh5os+Yu75ltfjQ8TkPx712Ydt/Zl/6TUPm7dLCGfK71IEHerDhhBa8B8h++/C3hq+aueFaLCoIBLn4/pBSCBZSXJLpkomoOI955YBbHma+dTc1nDL/oOuYvggJpHCd4pMczNNVb+WWEMW0rGgAROHWpwVDMfkNpeSwb65PcSXXlfELLTvctbPAhnYPUllivD268Ghcq6BmBp/g6Tu+tmDqFpnS8eFzXYzYFY0te/dgYDTrC7w+2cU4TUiGNwD38KiKVuPDMTBa4VQbI2RpV+fdn7TadtfRYH0XhUUK9MHuPb+pviCuxC2qzRaZ2JL8KAsIUz8WpNQOqN20AlJP7CyT3+KJfxV5DrX6F3GrZf23wOt8QRMHtz7F0Zn3lVOGSzNG61b1kfB0a1sYqH4TIrv7RsbbQM/bn+4ZcbxExMXqSCzbbbRRnyyhjsxu/FCtbzelRs5bCDcLmq7HCc+5uxGQqa3k6C6I57CDjjYggCLtF2WYNHShhEwHh+z4LWxbl

In [4]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get weather of the location"""
    return f"It's sunny at {location}"

model_with_tools = model.bind_tools([get_weather])


In [5]:
response = model_with_tools.invoke("What's the weather in Paris ?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_2898416': 'EtMDCtADARFNMg9l1kjLZWk0QJ4bAWndLzxi+zpcrWaZ8eyxdYc0k+QI03tDmckLr+99rn7mkoSwbRmsyccEcis9ucS6py1scxiy1rc7iIDQutmSbpHpWETGIY04D/UhLaaW47xoqAAGLxjIEzLEFk6cV7g2MctW37bucQt7qwbI2CtjCZaUGHhVIQ1Tw0o4iR/2lbmqAKelwizzVkCkz/aor1wMC61e62wvpPuHiPVm5jR1O4373tFfK9xOB5eEBBtn4W7ROFdNV5xWKsVuw5/KkHSpaM9h0yuQ+jYwr3dTwO7Mdt5EZJBADg6wjjK9dcK4d2ow6Hy4bxXMmiXL8pEBMFkfD9e0llKaMv+ZF2+P/EospY+c5zYuZe6uTArA5JmDpeBA7UkbLNtVmRZ3L7sEso5hDLNRXlRxUKj818vz64Tl5VfHHrCP4GNX5BzOl+2Yg03PKD+3NHiUUtzSX47Izf2qrGebzh+al6o381V6SHS5pYGmAh4GbCcuCwZwc1ifBE8QTbBlNdaDJgMZqnYpV9MWFtjLRkwuHz3OFJ7DYWdMN7BarQOFRFmCbBqm2NXlhWFtyMibhShvNNa6U7ZKsTqbewoq/MJfOb4Ntw11fo+VqhI='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a05929-8ae5-76c3-b9e1-d459fc7355d4-0' tool_ca

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools. invoke(messages)
messages. append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg. tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather . invoke(tool_call)
    messages . append (tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools . invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny.
